## Exercise 0 solution

1. Multiple Excel sheets in each file, each with a different name, but each file contains a `Table 1`.
2. The data does not start on row one. Headers are on row 7, followed by a blank line, followed by the actual data.
3. The data is stored in an inconvenient way, with ranks 1-50 in the first set of columns and ranks 51-100 in a second set of columns.
4. The second worksheet `2015boysnamesfinal.xlsx` contains extra columns between the data of interest, resulting in the second set of columns (ranks 51-100) being placed in a different position.
5. The year from which the data comes is only reported in the Excel file name, not within the data itself.
6. There are notes below the data.

These differences will make it more difficult to automate re-arranging the data since we have to write code that can handle different input formats.

## Exercise 1 solution

1. Write a function called `read_boys_names` that takes a file name as an argument and reads the worksheet containing "Table 1" from that file. Don't forget to skip the first 6 rows.

In [ ]:
read_boys_names <- function(file, sheet_name) {
  read_excel(
    path = file,
    sheet = get_data_sheet_name(file, term = sheet_name),
    skip = 6
  )
}

2. Test your function by using it to read *one* of the boys names Excel files.

In [ ]:
read_boys_names(boy_file_names[1], sheet_name = "Table 1") %>% glimpse()

3. Use the `map()` function to create a list of data frames called `boysNames` from all the Excel files, using the function you wrote in step 1.

In [ ]:
boysNames <- map(boy_file_names, read_boys_names, sheet_name = "Table 1")

## Exercise 2 solution

1. Write a function called `namecount` that takes a data frame as an argument and returns a modified version, which keeps only columns that include the strings `Name` and `Count` in the column names. HINT: see the `?matches` function. 

In [ ]:
  namecount <- function(data) {
      select(data, matches("Name|Count"))
  }

2. Test your function on the first data frame in the list of `boysNames` data.

In [ ]:
  namecount(boysNames[[1]])

3. Use the `map()` function to call your `namecount()` function on each data frame in the list called `boysNames`.Save the results back to the list called `boysNames`.

In [ ]:
  boysNames <- map(boysNames, namecount)

## Exercise 3 solution

1. Create a new function called `cleanupNamesData` that:

In [ ]:
cleanupNamesData <- function(file){

  # A) subsets data to include only those columns that include the term `Name` and `Count` and apply listwise deletion
  subsetted_file <- file %>%
    select(matches("Name|Count")) %>%
    drop_na()

  # B) subsets two separate data frames, with first and second set of `Name` and `Count` columns 
  first_columns <- select(subsetted_file, Name = Name...2, Count = Count...3) 

  second_columns <- select(subsetted_file, Name = matches("Name...6|Name...7|Name...8"),
                                           Count = matches("Count...7|Count...8|Count...9"))

  # C) appends the two datasets
  bind_rows(first_columns, second_columns)
}

# D) once you've written the function, test it out on *one* of the data frames in the list
boysNames[[2]] %>% glimpse() # before cleanup
boysNames[[2]] %>% cleanupNamesData() %>% glimpse() # after cleanup

2. Your task now is to use the `map()` function to apply each of these transformations to all the elements in `boysNames`. 

In [ ]:
# apply your function to elements in `boysNames` using `map()`
boysNames <- map(boysNames, cleanupNamesData)

## Exercise 4 solution

1. Turn the list of boys names data frames into a single data frame.

In [ ]:
# convert list of data frames into single data frame
boysNames <- bind_rows(boysNames)
glimpse(boysNames)

2. Create a new directory called `all` within `dataSets` and write the data to a `.csv` file. HINT: see the `?dir.create` and `?write_csv` functions.

In [ ]:
#| eval: false
dir.create("dataSets/all")

write_csv(boysNames, "dataSets/all/boys_names.csv")

3. What were the five most popular names in 2013?   

In [ ]:
boysNames %>% 
  filter(Year == 2013) %>%
  arrange(desc(Count)) %>%
  head()

4. How has the popularity of the name "ANDREW" changed over time?

In [ ]:
andrew <- filter(boysNames, Name == "ANDREW")

ggplot(andrew, aes(x = Year, y = Count)) +
    geom_line() +
    ggtitle("Popularity of Andrew, over time")